# Week 3 — Model Training and Selection
### Restaurant Demand Forecasting / Inventory Optimization

**Project:** AI Demand Forecasting — Store Sales Time Series  
**Dataset:** Kaggle Store Sales Time Series Forecasting  
**Goal:** Train and compare forecasting models — Baseline → Random Forest → XGBoost.  
Tune hyperparameters using Time-Series Cross-Validation and select the best model.

---

#### Notebook Flow
1. Load processed feature data from Week 2
2. Define features and target
3. Baseline — Linear Regression
4. Advanced Model — Random Forest Regressor
5. Advanced Model — XGBoost Regressor
6. Hyperparameter Tuning with Time-Series Cross-Validation
7. Final Model Evaluation on Validation Set
8. Forecast vs Actuals — Visual Comparison
9. Save Best Model

## 1. Setup and Load Data

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# ── Paths ──────────────────────────────────────────────────────────────────
project_root   = Path.cwd()
processed_dir  = project_root / 'data' / 'processed'
models_dir     = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

# ── Load feature files produced by Week 2 ─────────────────────────────────
# Week 2 saved four CSVs.  We load train and val; test is held out for Week 4.
train_feat = pd.read_csv(processed_dir / 'train_features.csv', parse_dates=['date'])
val_feat   = pd.read_csv(processed_dir / 'val_features.csv',   parse_dates=['date'])
test_feat  = pd.read_csv(processed_dir / 'test_features.csv',  parse_dates=['date'])

print('Train features :', train_feat.shape)
print('Val   features :', val_feat.shape)
print('Test  features :', test_feat.shape)
print()
print('Columns:', list(train_feat.columns))

## 2. Define Features and Target

We predict `total_sales` (raw daily sales).  
The model is trained on log1p-transformed sales to reduce the effect of large spikes,  
then predictions are converted back with `np.expm1` for evaluation.

In [ ]:
# Feature columns — everything except date and the two target columns
FEATURE_COLS = [
    'total_promo_items', 'transactions', 'oil_price',
    'year', 'month', 'day', 'dayofweek', 'weekofyear', 'quarter',
    'is_weekend', 'is_month_start', 'is_month_end',
    'is_holiday', 'holiday_level',
    'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'rolling_mean_7', 'rolling_std_14', 'rolling_max_7',
    'promo_lag_7', 'promo_rolling_7'
]

TARGET      = 'sales_log1p'   # train on log-scale
TARGET_RAW  = 'total_sales'   # evaluate on raw scale

# Confirm all feature columns are present
missing_cols = [c for c in FEATURE_COLS if c not in train_feat.columns]
if missing_cols:
    raise ValueError(f'Missing feature columns: {missing_cols}')

X_train = train_feat[FEATURE_COLS]
y_train = train_feat[TARGET]
y_train_raw = train_feat[TARGET_RAW]

X_val   = val_feat[FEATURE_COLS]
y_val   = val_feat[TARGET]
y_val_raw = val_feat[TARGET_RAW]

X_test  = test_feat[FEATURE_COLS]
y_test_raw = test_feat[TARGET_RAW]

print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}  |  y_val  : {y_val.shape}')
print(f'X_test : {X_test.shape}')

## 3. Helper — Evaluation Function

A single function to compute MAE and RMSE on the raw (un-logged) scale  
so all models are compared on the same, interpretable metric.

In [ ]:
def evaluate(model_name: str,
             y_true_raw: pd.Series,
             y_pred_log: np.ndarray) -> dict:
    """
    Convert log-scale predictions back to raw sales, then compute MAE and RMSE.
    Returns a results dict and prints a summary.
    """
    y_pred_raw = np.expm1(y_pred_log)          # inverse of log1p
    y_pred_raw = np.clip(y_pred_raw, 0, None)  # no negative sales

    mae  = mean_absolute_error(y_true_raw, y_pred_raw)
    rmse = np.sqrt(mean_squared_error(y_true_raw, y_pred_raw))

    print(f'  {model_name:<30}  MAE = {mae:>12,.2f}   RMSE = {rmse:>12,.2f}')
    return {'model': model_name, 'MAE': mae, 'RMSE': rmse,
            'y_pred_raw': y_pred_raw}


results = []   # collect all model results for final comparison

## 4. Baseline — Linear Regression

We start with the simplest model to establish a performance floor.  
Any advanced model that cannot beat this baseline is not worth deploying.

In [ ]:
# Scale features for Linear Regression (tree models don't need this)
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_vl_sc = scaler.transform(X_val)

lr = LinearRegression()
lr.fit(X_tr_sc, y_train)

lr_pred_log = lr.predict(X_vl_sc)

print('── Baseline: Linear Regression ──────────────────────────────')
res_lr = evaluate('Linear Regression', y_val_raw, lr_pred_log)
results.append(res_lr)

## 5. Random Forest Regressor

Random Forest handles non-linear relationships and is robust to outliers.  
We first run a quick default-parameter version, then tune it in Section 7.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

rf_pred_log = rf.predict(X_val)

print('── Random Forest (default params) ───────────────────────────')
res_rf = evaluate('Random Forest (default)', y_val_raw, rf_pred_log)
results.append(res_rf)

## 6. XGBoost Regressor

XGBoost is typically stronger than Random Forest on tabular time-series data  
because it learns sequentially from residual errors.  
We run a default version first, tune it in Section 7.

In [ ]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_pred_log = xgb.predict(X_val)

print('── XGBoost (default params) ──────────────────────────────────')
res_xgb = evaluate('XGBoost (default)', y_val_raw, xgb_pred_log)
results.append(res_xgb)

## 7. Hyperparameter Tuning with Time-Series Cross-Validation

We use `TimeSeriesSplit` — never shuffle time-series data for cross-validation.  
Random splits would leak future data into training, producing artificially low errors.

We tune XGBoost (the stronger model) with a manual grid to keep runtime manageable.

In [ ]:
# ── Combine train + val for cross-validation ───────────────────────────────
# The test set is never touched here.
X_cv = pd.concat([X_train, X_val], ignore_index=True)
y_cv = pd.concat([y_train, y_val], ignore_index=True)
y_cv_raw = pd.concat([y_train_raw, y_val_raw], ignore_index=True)

tscv = TimeSeriesSplit(n_splits=5)

# Parameter grid to search
param_grid = [
    {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8},
    {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.05, 'subsample': 0.8},
    {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8},
    {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 0.9},
    {'n_estimators': 700, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 0.9},
]

print(f'Running {len(param_grid)} parameter sets × {tscv.n_splits} folds ...\n')

cv_results = []

for i, params in enumerate(param_grid):
    fold_maes = []
    for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X_cv)):
        Xtr, Xvl = X_cv.iloc[tr_idx], X_cv.iloc[vl_idx]
        ytr, yvl = y_cv.iloc[tr_idx], y_cv.iloc[vl_idx]
        yvl_raw  = y_cv_raw.iloc[vl_idx]

        model = XGBRegressor(
            **params,
            colsample_bytree=0.8,
            objective='reg:squarederror',
            n_jobs=-1,
            random_state=42,
            verbosity=0
        )
        model.fit(Xtr, ytr, verbose=False)
        pred_raw = np.expm1(model.predict(Xvl))
        pred_raw = np.clip(pred_raw, 0, None)
        fold_maes.append(mean_absolute_error(yvl_raw, pred_raw))

    mean_mae = np.mean(fold_maes)
    cv_results.append({**params, 'cv_mae': mean_mae})
    print(f'  Params set {i+1}: {params}  →  CV MAE = {mean_mae:,.2f}')

cv_df = pd.DataFrame(cv_results).sort_values('cv_mae')
print('\n── CV Results (sorted by MAE) ──────────────────────────────')
print(cv_df.to_string(index=False))

## 8. Train Final Tuned XGBoost on Full Training Data

We pick the best hyperparameters from CV and retrain on the entire training set.

In [ ]:
best_params = cv_df.iloc[0].drop('cv_mae').to_dict()
# n_estimators and max_depth were stored as float by pandas — cast them back
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth']    = int(best_params['max_depth'])

print('Best hyperparameters:', best_params)

xgb_tuned = XGBRegressor(
    **best_params,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_tuned_pred_log = xgb_tuned.predict(X_val)

print('\n── XGBoost (tuned) ────────────────────────────────────────────')
res_xgb_tuned = evaluate('XGBoost (tuned)', y_val_raw, xgb_tuned_pred_log)
results.append(res_xgb_tuned)

## 9. Model Comparison Summary

In [ ]:
summary = pd.DataFrame([{'Model': r['model'], 'MAE': r['MAE'], 'RMSE': r['RMSE']}
                         for r in results])
summary = summary.sort_values('MAE').reset_index(drop=True)

print('═' * 60)
print('  MODEL COMPARISON — Validation Set')
print('═' * 60)
print(summary.to_string(index=False))
print('═' * 60)
print(f'  Best model : {summary.iloc[0]["Model"]}')
print(f'  Best MAE   : {summary.iloc[0]["MAE"]:,.2f}')
print(f'  Best RMSE  : {summary.iloc[0]["RMSE"]:,.2f}')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['MAE', 'RMSE']):
    bars = ax.barh(summary['Model'], summary[metric],
                   color=sns.color_palette('muted', len(summary)))
    ax.set_xlabel(metric)
    ax.set_title(f'Model Comparison — {metric} (lower is better)')
    ax.bar_label(bars, fmt='%.0f', padding=4)

plt.tight_layout()
plt.savefig(processed_dir / 'week3_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Forecast vs Actuals — Visual Comparison

Plot the validation-period predictions of all models against actual daily sales.  
This is the most intuitive way to see which model tracks demand patterns best.

In [ ]:
val_dates = val_feat['date']

fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(val_dates, y_val_raw.values,
        color='black', linewidth=1.8, label='Actual Sales', zorder=5)

colors = ['#e07b54', '#5b9bd5', '#70ad47']
line_styles = ['--', '-.', ':']

for res, color, ls in zip(
        [res_lr, res_rf, res_xgb_tuned],
        colors, line_styles):
    ax.plot(val_dates, res['y_pred_raw'],
            color=color, linewidth=1.4,
            linestyle=ls, alpha=0.85,
            label=res['model'])

ax.set_title('Validation Set — Forecast vs Actual Daily Sales',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(processed_dir / 'week3_forecast_vs_actuals.png', dpi=100, bbox_inches='tight')
plt.show()

## 11. Residuals Analysis

Check if the best model's errors are random (good) or show a pattern (bad — means the model is missing something systematic).

In [ ]:
best_pred_raw = res_xgb_tuned['y_pred_raw']
residuals     = y_val_raw.values - best_pred_raw

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Residuals over time
axes[0].plot(val_dates, residuals, color='steelblue', linewidth=0.9, alpha=0.8)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.2)
axes[0].set_title('Residuals over Time — XGBoost (tuned)')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Actual − Predicted')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Residual distribution
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.2)
axes[1].set_title('Residual Distribution — XGBoost (tuned)')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig(processed_dir / 'week3_residuals.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Mean residual  : {residuals.mean():,.2f}  (close to 0 = unbiased)')
print(f'Std of residual: {residuals.std():,.2f}')

## 12. Learning Curve — XGBoost Training vs Validation Loss

Visualise how train error and validation error evolve with number of boosting rounds.  
This helps detect overfitting (train error falls but val error rises or plateaus).

In [ ]:
# Retrain with eval_metric to capture per-round scores
xgb_lc = XGBRegressor(
    **best_params,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    eval_metric='rmse',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

eval_results = {}
xgb_lc.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
    callbacks=[
        # XGBoost stores eval results internally; we extract after fitting
    ]
)
evals = xgb_lc.evals_result()

train_rmse = evals['validation_0']['rmse']
val_rmse   = evals['validation_1']['rmse']
rounds     = range(1, len(train_rmse) + 1)

plt.figure(figsize=(10, 4))
plt.plot(rounds, train_rmse, label='Train RMSE (log scale)', color='steelblue', linewidth=1.2)
plt.plot(rounds, val_rmse,   label='Val RMSE (log scale)',   color='orange',    linewidth=1.4)
plt.xlabel('Boosting Round')
plt.ylabel('RMSE (log-scale target)')
plt.title('XGBoost Learning Curve — Train vs Validation RMSE')
plt.legend()
plt.tight_layout()
plt.savefig(processed_dir / 'week3_learning_curve.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Min val RMSE at round : {int(np.argmin(val_rmse)) + 1}')
print(f'Min val RMSE value    : {min(val_rmse):.5f}')

## 13. Feature Importance

Understand which features the tuned XGBoost model relies on most.  
This is both a model diagnostic and a business communication tool for Week 4.

In [ ]:
feat_imp = pd.Series(
    xgb_tuned.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=feat_imp.values,
    y=feat_imp.index,
    palette='Blues_r'
)
plt.title('XGBoost (tuned) — Feature Importance', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(processed_dir / 'week3_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nTop 5 most important features:')
print(feat_imp.head(5).to_string())

## 14. Save Best Model

Serialize the tuned XGBoost model to disk so Week 4 can load it directly  
for final test-set evaluation and business reporting.

> ⚠️ **Note:** The saved `.pkl` model file should be added to `.gitignore`  
> as per the internship security guidelines.  
> Only the notebook code and evaluation charts should be committed.

In [ ]:
model_path  = models_dir / 'xgboost_tuned_week3.pkl'
scaler_path = models_dir / 'scaler_week3.pkl'

joblib.dump(xgb_tuned, model_path)
joblib.dump(scaler,    scaler_path)

print('Model saved to  :', model_path)
print('Scaler saved to :', scaler_path)
print()
print('⚠️  Add models/*.pkl to .gitignore — do not push model weights to GitHub')

## 15. Week 3 Summary

Print a final summary of what was achieved this week.

In [ ]:
best = summary.iloc[0]

print('═' * 62)
print('  WEEK 3 — FINAL SUMMARY')
print('═' * 62)
print()
print('  Models trained  : Linear Regression, Random Forest, XGBoost')
print('  Tuning method   : TimeSeriesSplit (5 folds), manual grid')
print('  Evaluation set  : Validation set (Jan – May 2017)')
print()
print(f'  Best model      : {best["Model"]}')
print(f'  Best Val MAE    : {best["MAE"]:,.2f}')
print(f'  Best Val RMSE   : {best["RMSE"]:,.2f}')
print()
print('  Outputs saved to data/processed/')
print('    week3_model_comparison.png')
print('    week3_forecast_vs_actuals.png')
print('    week3_residuals.png')
print('    week3_learning_curve.png')
print('    week3_feature_importance.png')
print()
print('  Next step: Week 4 — Final test-set evaluation,')
print('  feature importance business report, forecast dashboard')
print('═' * 62)